In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
from ultralytics import YOLO

PROJECT_PATH = Path("../")

DATASET_PATH = PROJECT_PATH / "02_datasets" / "DUT_Anti_UAV"

MODEL_PATH = (
    PROJECT_PATH
    / "runs"
    / "detect"
    / "04_experiments"
    / "EXP001_YOLOv8s_baseline"
    / "weights"
    / "best.pt"
)

TEST_IMAGES = DATASET_PATH / "Images" / "test"
TEST_LABELS = DATASET_PATH / "labels" / "test"

print("Dataset:", DATASET_PATH.exists())
print("Model:", MODEL_PATH.exists())

Dataset: True
Model: True


In [2]:
gt_records = []

for label_path in sorted(TEST_LABELS.glob("*.txt")):
    
    image_name = label_path.stem
    image_path = TEST_IMAGES / f"{image_name}.jpg"
    
    with Image.open(image_path) as im:
        img_w, img_h = im.size

    with open(label_path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    for gt_idx, line in enumerate(lines):

        values = line.split()

        cls = int(values[0])
        xc = float(values[1])
        yc = float(values[2])
        w = float(values[3])
        h = float(values[4])

        x1 = (xc - w / 2) * img_w
        y1 = (yc - h / 2) * img_h
        x2 = (xc + w / 2) * img_w
        y2 = (yc + h / 2) * img_h

        area = w * h

        if area < 0.001:
            size_category = "Tiny"
        elif area < 0.01:
            size_category = "Small"
        elif area < 0.1:
            size_category = "Medium"
        else:
            size_category = "Large"

        gt_records.append({
            "image": image_name,
            "gt_id": gt_idx,
            "class": cls,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "area": area,
            "size_category": size_category
        })

gt_clean = pd.DataFrame(gt_records)

print("Total GT UAVs:", len(gt_clean))
print()
print(gt_clean["size_category"].value_counts())

Total GT UAVs: 2245

size_category
Tiny      1148
Small      555
Medium     467
Large       75
Name: count, dtype: int64


In [3]:
from ultralytics import YOLO
import pandas as pd
from pathlib import Path
import torch

torch.cuda.empty_cache()

model = YOLO(MODEL_PATH)

pred_records = []

results = model.predict(
    source=str(TEST_IMAGES),
    imgsz=640,
    conf=0.25,
    iou=0.7,
    device=0,
    stream=True,
    verbose=False
)

for result in results:

    # اسم الصورة الحقيقي الذي عالجه YOLO
    image_name = Path(result.path).stem

    for pred_id, box in enumerate(result.boxes):

        xyxy = box.xyxy[0].cpu().numpy()

        pred_records.append({
            "image": image_name,
            "pred_id": pred_id,
            "x1": float(xyxy[0]),
            "y1": float(xyxy[1]),
            "x2": float(xyxy[2]),
            "y2": float(xyxy[3]),
            "confidence": float(box.conf[0]),
            "class": int(box.cls[0])
        })

pred_clean = pd.DataFrame(pred_records)

print("Predicted boxes:", len(pred_clean))
print("Images with predictions:", pred_clean["image"].nunique())

pred_clean.head()

Predicted boxes: 2263
Images with predictions: 2114


,image,pred_id,x1,y1,x2,y2,confidence,class
0,00001,0,614.140015,541.470337,682.293091,602.050659,0.877906,0
1,00002,0,512.551147,226.074768,572.945312,284.010437,0.873486,0
2,00003,0,472.303223,77.581154,779.630249,190.478180,0.862339,0
3,00004,0,782.170227,67.874146,896.035828,118.388596,0.874388,0
4,00005,0,665.116089,353.410828,690.285034,373.290710,0.690354,0


In [4]:
print("Total GT UAVs:", len(gt_clean))
print("Predicted boxes:", len(pred_clean))
print("Images with predictions:", pred_clean["image"].nunique())
print("Test images:", len(list(TEST_IMAGES.glob("*.jpg"))))

Total GT UAVs: 2245
Predicted boxes: 2263
Images with predictions: 2114
Test images: 2200


In [2]:
from pathlib import Path

print("Current folder:", Path.cwd())

weights = list(Path.cwd().rglob("best.pt"))

for weight in weights:
    print(weight)

Current folder: d:\master\Master_Drone_Detection\notebooks
